# 04 - Analyse SQL

Objectif : structurer les données dans SQLite et valider les KPI avec SQL. Le dashboard Power BI utilise directement `incident_clean.csv` et des mesures DAX dynamiques.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATABASE_DIR = PROJECT_ROOT / 'data' / 'database'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
DATA_PATH = PROCESSED_DIR / 'incident_clean.csv'
DB_PATH = DATABASE_DIR / 'incident.db'

df = pd.read_csv(DATA_PATH, low_memory=False)
conn = sqlite3.connect(DB_PATH)
df.to_sql('incidents', conn, if_exists='replace', index=False)

print(f'Base créée : {DB_PATH}')

Base créée : c:\Users\djami\Desktop\Projects\IT-Service-Performance-Analytics\data\database\incident.db


In [3]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

# Les résultats SQL restent dans le notebook : aucun CSV de synthèse n'est requis.
# Power BI calcule les mêmes KPI depuis incident_clean.csv avec des mesures DAX.

## KPI globaux

In [4]:
kpi_sql = run_sql('''
SELECT
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h,
    ROUND(AVG(closure_time_hours), 2) AS cloture_moyenne_h,
    ROUND(AVG(reassignment_count), 2) AS reassignations_moyennes,
    ROUND(AVG(reopen_count), 2) AS reouvertures_moyennes
FROM incidents;
''')

kpi_sql

,incidents,sla_pct,resolution_moyenne_h,cloture_moyenne_h,reassignations_moyennes,reouvertures_moyennes
0,24918,63.45,178.17,315.33,0.94,0.01


## Performance par groupe support

In [5]:
groupe_sql = run_sql('''
SELECT
    assignment_group,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h
FROM incidents
GROUP BY assignment_group
HAVING COUNT(*) >= 20
ORDER BY incidents DESC
LIMIT 10;
''')

groupe_sql

,assignment_group,incidents,sla_pct,resolution_moyenne_h
0,Group 70,9444,83.84,50.22
1,Inconnu,2157,48.63,106.37
2,Group 25,1243,42.80,251.94
3,Group 39,1199,68.81,71.96
4,Group 24,1060,64.81,136.59
5,Group 23,811,58.69,245.22
6,Group 64,716,89.11,18.75
7,Group 73,576,53.99,124.71
8,Group 28,545,53.58,135.38
9,Group 27,518,58.30,123.84


## SLA par priorité

In [6]:
priorite_sql = run_sql('''
SELECT
    priority,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h
FROM incidents
GROUP BY priority
ORDER BY priority;
''')

priorite_sql

,priority,incidents,sla_pct,resolution_moyenne_h
0,1 - Critical,270,2.22,266.08
1,2 - High,408,0.98,152.46
2,3 - Moderate,23466,64.56,174.35
3,4 - Low,774,84.11,283.19


## Volume mensuel

In [7]:
mois_sql = run_sql('''
SELECT
    opened_month,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct
FROM incidents
GROUP BY opened_month
ORDER BY opened_month;
''')

mois_sql

,opened_month,incidents,sla_pct
0,2016-02,207,64.73
1,2016-03,8995,39.87
2,2016-04,7934,76.71
3,2016-05,7508,77.58
4,2016-06,5,80.00
5,2016-07,14,78.57
6,2016-08,15,46.67
7,2016-09,12,58.33
8,2016-10,16,43.75
9,2016-11,26,65.38


## Effet des réassignations

In [8]:
reassignment_sql = run_sql('''
SELECT
    CASE WHEN has_reassignment = 1 THEN 'Avec réassignation' ELSE 'Sans réassignation' END AS reassignment_status,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h
FROM incidents
GROUP BY has_reassignment
ORDER BY has_reassignment;
''')

reassignment_sql

,reassignment_status,incidents,sla_pct,resolution_moyenne_h
0,Sans réassignation,13549,78.39,94.98
1,Avec réassignation,11369,45.64,266.08


## Incidents les plus longs

Cette requête permet d'identifier les incidents avec les temps de résolution les plus élevés.

In [9]:
top_long_incidents_sql = run_sql('''
SELECT
    number,
    assignment_group,
    category,
    priority,
    resolution_time_hours
FROM incidents
WHERE resolution_time_hours IS NOT NULL
ORDER BY resolution_time_hours DESC
LIMIT 25;
''')

top_long_incidents_sql

,number,assignment_group,category,priority,resolution_time_hours
0,INC0001839,Group 31,Category 45,3 - Moderate,8070.17
1,INC0001349,Group 67,Category 45,4 - Low,8020.27
2,INC0007343,Group 31,Category 46,3 - Moderate,7803.68
3,INC0001881,Group 35,Category 55,4 - Low,7316.45
4,INC0001978,Group 35,Category 55,4 - Low,7313.93
5,INC0001984,Group 35,Category 55,4 - Low,7313.87
6,INC0019986,Group 3,Category 42,3 - Moderate,7220.58
7,INC0000343,Group 31,Category 45,4 - Low,7122.28
8,INC0000307,Group 66,Category 45,4 - Low,7056.42
9,INC0003982,Group 47,Category 37,3 - Moderate,6729.00


In [10]:
conn.close()

## Conclusion

Les requêtes SQL fournissent un contrôle structuré des indicateurs IT calculés dans Power BI.

Les résultats confirment trois axes de pilotage : surveiller le SLA global, isoler les priorités critiques et hautes dont la conformité SLA est très faible, et suivre les groupes support qui combinent volume élevé et délais de résolution importants. La tendance mensuelle doit être interprétée avec prudence, car la période observée est très déséquilibrée en volume.